<a href="https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. My lane (or freestyle) and why

Lane 4, CTR / Engagement Opportunity Scoring. Which visible pages get fewer clicks than their position would suggest, and are worth a review?

Notebook 01 showed CTR falls off hard by position tier. So raw CTR on its own doesn't tell you much. A page sitting at position 3 with mediocre CTR could be doing worse for its slot than a page at position 15 with lower CTR outright. What actually matters is the gap against other pages in the same tier. That's measurable from what's already in the data, and it points at something a person can go and fix.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.isdir("flyrank-ml-internship-starter"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "flyrank-ml-internship-starter"], check=True)
if IN_COLAB:
    os.chdir("/content/flyrank-ml-internship-starter")
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages loaded")


30,000 pages loaded


## 2. The question: decision, action, cost of a wrong call
Decision: which pages go on the review queue this week, and in what order.

Who acts: a content or SEO person. They check the title and meta against what the page actually delivers, and rewrite if there's a mismatch. Cheap, reversible.

Cost of a wrong call: a false positive wastes an hour on a page that was fine. A false negative is quieter and worse, since a struggling page just stays invisible. The failure I most want to avoid is flagging low-volume pages where the CTR is really just noise, because that burns trust in the whole queue.

In [2]:
low_vol = (df["impressions_90d"] < 100).mean()
print(f"pages under 100 impressions in 90d: {low_vol:.1%}")
print(f"weekly capacity of 20 pages covers {20 / len(df):.2%} of the inventory")
print(f"at 20 pages/week, one full pass takes {len(df) / 20 / 52:.0f} years")

pages under 100 impressions in 90d: 26.6%
weekly capacity of 20 pages covers 0.07% of the inventory
at 20 pages/week, one full pass takes 29 years


## 3. Quick look at the data (2-3 real numbers)

Filtered to pages with at least 100 impressions in 90 days, which keeps 22,006 of 30,000.

Median CTR varies by tier (0.23 on page_1, 0.06 on page_3_5), but the spread inside each tier is wider than the gaps between them. Page_1 runs 0.09 to 0.46 across the middle half, an IQR of 0.37 against a median of 0.23. So pages sitting at the same position differ more from each other than the tiers differ on average. That within-tier gap is what's worth scoring.

Two things I noticed and won't paper over. The tiers don't order cleanly, since top_3 medians lower than page_1, so I can't lean on tier rank as a proxy for position. And the deep tier is zero at every quartile, so it can't support gap scoring and I'd exclude it.

In [3]:
visible = df[df["impressions_90d"] >= 100]
ctr_by_tier = visible.groupby("position_tier")["ctr"].agg(["median", "count"])
print(ctr_by_tier.round(3).to_string())
spread = visible.groupby("position_tier")["ctr"].quantile([0.25, 0.75]).unstack()
spread["iqr"] = spread[0.75] - spread[0.25]
print(spread.round(3).to_string())
print(f"kept {len(visible):,} of {len(df):,} pages after the 100-impression filter")


               median  count
position_tier               
deep             0.00    879
page_1           0.23   8633
page_3_5         0.06   6058
striking         0.15   5903
top_3            0.19    533
               0.25  0.75   iqr
position_tier                  
deep           0.00  0.00  0.00
page_1         0.09  0.46  0.37
page_3_5       0.00  0.19  0.19
striking       0.00  0.34  0.34
top_3          0.05  0.48  0.43
kept 22,006 of 30,000 pages after the 100-impression filter


## 4. Careful words: what I can and can't claim

What I can say: observed and directional. I can measure how far a page's CTR sits below others in the same position tier, rank pages by that gap, and hand someone a queue with a reason attached to each row. Decision support, not much more.

What I can't say: anything causal. Every content_id appears exactly once, and there are no dates or before/after fields, so I only ever see one snapshot. I can't watch a page change, which means I can never show that a rewrite caused a recovery. A big gap says "worth a look", not "fix this and clicks go up".

I also won't claim anything about how Google ranks pages. I'm looking at outcomes, not at the algorithm that produced them.

One more limit: 32 clients, median 567 pages each, but the biggest has 7,008. So a pattern could easily be one client's site conventions rather than something general. Any result needs a per-client check before I trust it.

In [4]:
print("columns available:", len(df.columns))
print("has any before/after refresh field:", any("after" in c or "post" in c or "refresh_date" in c for c in df.columns))
print("has any date column:", any(df[c].dtype == "object" and "date" in c.lower() for c in df.columns))
print("rows per content_id:", df["content_id"].value_counts().max())
print("distinct clients:", df["client_id"].nunique())
print(f"pages per client: median {df['client_id'].value_counts().median():.0f}, max {df['client_id'].value_counts().max():,}")


columns available: 44
has any before/after refresh field: False
has any date column: False
rows per content_id: 1
distinct clients: 32
pages per client: median 567, max 7,008


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.